# Week 5b -- Design Trends Analysis

This notebook analyzes `design_trends.csv`, which was generated by `design_trends.py` using the Wikipedia Pageviews API. Each row represents one HCI or UX-related Wikipedia article and summarizes 90 days of daily traffic data from January 30 to April 30, 2026.

The goal is to understand which topics are genuinely rising in public interest, which have peaked, and which are fading -- using traffic as a proxy for cultural attention.

**Before running:** make sure `design_trends.csv` is in the same folder as this notebook. If not, run `python design_trends.py` first.

## Setup

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(".")
TRENDS_CSV = DATA_DIR / "design_trends.csv"

assert TRENDS_CSV.is_file(), (
    f"Missing {TRENDS_CSV.resolve()}. Run: python design_trends.py"
)

print("pandas:", pd.__version__)

---
## 1. What does the dataset look like?

`head()` shows the first few rows so we can see what fields exist and what the top-ranked articles are. `info()` tells us the data types and whether any columns have missing values.

In [ ]:
df = pd.read_csv(TRENDS_CSV, encoding="utf-8")

# The top rows are the most-viewed HCI topics on Wikipedia in the last 90 days.
# Rank 1 having the most views tells us what the public is most curious about
# in the HCI and UX space right now.
df.head(10)

In [ ]:
# info() confirms column types and whether anything is missing.
# If a column shows fewer non-null values than the total rows,
# that means some articles did not have enough data for that calculation.
df.info()

---
## 2. How is traffic distributed across all 50 articles?

`describe()` gives us the summary statistics for total views. This tells us whether traffic is spread evenly or concentrated in a few top articles.

In [ ]:
# The gap between the mean and the max tells us how concentrated traffic is.
# If the max is much higher than the mean, a small number of articles
# are pulling most of the public attention -- which is typical of how
# cultural interest works. Most topics get ignored; a few dominate.
df["total_views_in_window"].describe()

In [ ]:
# value_counts() on the momentum column shows how many articles
# are rising, peaked/stable, or fading right now.
# This gives a quick read on the overall health of public interest
# in HCI topics -- is curiosity growing or declining as a whole?
df["momentum"].value_counts()

---
## 3. Which topics are currently rising in public interest?

We filter to only the articles with Rising momentum -- meaning their traffic in the most recent 30 days was at least 10% higher than the 30 days before that. These are the topics the public is becoming more curious about right now.

In [ ]:
# Filtering to Rising articles only.
# The pct_change column tells us exactly how much traffic grew.
# A topic with 100% change means it doubled in views -- that is a strong signal,
# not just noise. A topic with 10-15% change is growing but modestly.
rising = df[df["momentum"] == "Rising"].copy()

print(f"{len(rising)} out of {len(df)} tracked articles are currently Rising.")
rising[["rank", "trend", "total_views_in_window", "pct_change_recent_vs_prior_30d"]].sort_values(
    "pct_change_recent_vs_prior_30d", ascending=False
)

---
## 4. Does momentum correlate with total traffic volume?

We group by momentum category and take the mean of total views. This answers: are rising topics generally the most-viewed ones, or can a low-traffic topic still be growing?

In [ ]:
# Grouping by momentum and averaging total views.
# If Rising articles have lower average views than Fading ones,
# it means new topics are emerging from obscurity -- which is interesting
# because it suggests the next big thing in HCI discourse might currently
# have very low absolute traffic but be growing fast.
mean_views_by_momentum = (
    df.groupby("momentum", as_index=False)["total_views_in_window"]
    .mean()
    .rename(columns={"total_views_in_window": "mean_total_views"})
    .sort_values("mean_total_views", ascending=False)
)
mean_views_by_momentum

---
## 5. Are there any missing values or data quality issues?

Some articles may not have had enough days of data for the momentum calculation to run. We check which columns are incomplete and what that means for the analysis.

In [ ]:
# isnull().sum() counts how many rows are missing a value in each column.
# If pct_change or recent_30d_avg_views has missing values, it means
# those articles did not have enough data for a 60-day comparison.
# That is a real data quality issue -- it means the momentum label
# for those rows may not be reliable.
missing = df.isnull().sum()
print("Columns with missing values:")
print(missing[missing > 0])
print()
print("Total rows:", len(df))
print("All columns complete:", missing.sum() == 0)

In [ ]:
# Check which specific articles have fewer than 90 days of data.
# An article with only 80 days means Wikipedia had no recorded views
# on some days -- possibly because the page had zero traffic,
# not because the data is broken.
incomplete = df[df["days_with_data"] < 90][["rank", "trend", "days_with_data", "momentum"]]
print(f"{len(incomplete)} articles have fewer than 90 days of data:")
incomplete